In [1]:
import torch
import os
import torchvision.transforms as transforms
import torchvision.datasets as datasets
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.quantized.engine = 'qnnpack'
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
from torch.ao.quantization import QConfigMapping


In [1]:
model_path = "pruned_Model_entire.pth"
Model = torch.load(model_path, map_location=device, weights_only=False)
Model.eval()

print("Model loaded successfully!")

NameError: name 'torch' is not defined

In [4]:
original_model = Model
for name, module in original_model.named_modules():
    if hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
        # Regular float layer
        print(f"{name} weight dtype: {module.weight.dtype}")
    elif hasattr(module, "weight") and callable(module.weight):
        # Quantized layer
        print(f"{name} weight dtype: {module.weight().dtype}")


features.0.0 weight dtype: torch.float32
features.0.1 weight dtype: torch.float32
features.1.conv.0.0 weight dtype: torch.float32
features.1.conv.0.1 weight dtype: torch.float32
features.1.conv.1 weight dtype: torch.float32
features.1.conv.2 weight dtype: torch.float32
features.2.conv.0.0 weight dtype: torch.float32
features.2.conv.0.1 weight dtype: torch.float32
features.2.conv.1.0 weight dtype: torch.float32
features.2.conv.1.1 weight dtype: torch.float32
features.2.conv.2 weight dtype: torch.float32
features.2.conv.3 weight dtype: torch.float32
features.3.conv.0.0 weight dtype: torch.float32
features.3.conv.0.1 weight dtype: torch.float32
features.3.conv.1.0 weight dtype: torch.float32
features.3.conv.1.1 weight dtype: torch.float32
features.3.conv.2 weight dtype: torch.float32
features.3.conv.3 weight dtype: torch.float32
features.4.conv.0.0 weight dtype: torch.float32
features.4.conv.0.1 weight dtype: torch.float32
features.4.conv.1.0 weight dtype: torch.float32
features.4.conv.1.

In [6]:
# quantization backend (choose 'fbgemm' for x86/CPU, 'qnnpack' for ARM/mobile)
torch.backends.quantized.engine = 'qnnpack'  
qconfig = get_default_qconfig(torch.backends.quantized.engine)


In [7]:
Calibration_transform = transforms.Compose([
    transforms.Resize (size = (224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Calibration_transform  = transforms.Compose([ # for the grayscale (Xray / MRI) datasets
#     transforms.Lambda(lambda img: img.convert("RGB")),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225]),
#                                               ])


val_dir = "colon_after_splitting/val"
# val_dir = "COVID19+PNEUMONIA+NORMAL Chest X-Ray Image Dataset/val"
# val_dir = "K+F MRI (clean)/val"
calibration_dataset = datasets.ImageFolder(val_dir, transform=Calibration_transform)
calibration_subset, _ = torch.utils.data.random_split(calibration_dataset, [200, len(calibration_dataset) - 200])
calibration_loader = torch.utils.data.DataLoader(calibration_subset, batch_size=32, shuffle=False)


In [ ]:
example_inputs, _ = next(iter(calibration_loader))
example_inputs = example_inputs.to(device)

In [ ]:
qconfig_mapping = QConfigMapping().set_global(qconfig) 
prepared_model = prepare_fx(Model, qconfig_mapping, example_inputs)

In [ ]:
print("🔄 Running calibration ...")
prepared_model.eval()
with torch.no_grad():
    for images, _ in calibration_loader:
        images = images.to(device)
        prepared_model(images)
print("✅ Calibration done!")


In [ ]:
quantized_model = convert_fx(prepared_model)
quantized_model.eval()
print("✅ Model quantized successfully!")

In [ ]:
# printing the unquantized 32FP
for name, param in quantized_model.named_parameters():
    print(f"{name} dtype: {param.dtype}")


In [ ]:
for name, module in quantized_model.named_modules():
    if hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
        # Regular float layer
        print(f"{name} weight dtype: {module.weight.dtype}")
    elif hasattr(module, "weight") and callable(module.weight):
        # Quantized layer
        print(f"{name} weight dtype: {module.weight().dtype}")

In [ ]:
# @title
import torch
import os

def print_model_info(model):
    total_params = 0
    total_qparams = 0
    total_fparams = 0

    print("===== Quantized Model Summary =====")
    for name, module in model.named_modules():
        # Skip top-level modules
        if len(list(module.children())) > 0:
            continue

        layer_type = type(module).__name__
        n_params = 0
        weight_dtype_str = "N/A"

        if hasattr(module, "weight"):
            # Quantized layer (weight is callable)
            if callable(module.weight):
                weight_tensor = module.weight()
                weight_dtype_str = str(weight_tensor.dtype)
                n_params = weight_tensor.numel()
                if weight_tensor.dtype in [torch.qint8, torch.quint8]:
                    total_qparams += n_params
                else:
                    total_fparams += n_params
            # Float layer
            elif isinstance(module.weight, torch.nn.Parameter):
                weight_tensor = module.weight
                weight_dtype_str = str(weight_tensor.dtype)
                n_params = weight_tensor.numel()
                total_fparams += n_params

        total_params += n_params
        print(f"{name:40s} | {layer_type:20s} | weight dtype: {weight_dtype_str:10s} | params: {n_params}")

    print("===================================")
    print(f"Total parameters        : {total_params}")
    print(f" - Quantized parameters  : {total_qparams}")
    print(f" - Float parameters      : {total_fparams}")

    if model_path and os.path.exists(model_path):
        size_mb = os.path.getsize(model_path) / 1e6
        print(f"Model file size: {size_mb:.2f} MB")


# Model Evaluation

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import random
from PIL import Image
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
from pathlib import Path


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
#& walk through the dataset directory and print number of images in each subdirectory

def walk_through_dir(dir_path):
    for dirpath, dirname, filenames in os.walk(dir_path):
        print(f"There are {len(dirname)} directories and {len(filenames)} images in '{dirpath}'")

dataset_path = "/colon_after_splitting"
# dataset_path = "/COVID19+PNEUMONIA+NORMAL Chest X-Ray Image Dataset"
# dataset_path = "K+F MRI (clean)"
walk_through_dir(dataset_path) 

In [ ]:
test_dir  = os.path.join(dataset_path, "test")
test_dir

In [ ]:
# test_transform = transforms.Compose([
#     transforms.Lambda(lambda img: img.convert("RGB")),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                         std=[0.229, 0.224, 0.225]),
# ])

# for RGB 
test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

In [ ]:
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
print(test_dataset)
print("-----------------------------------------------------------------------------")
print("Classes:", test_dataset.class_to_idx)
print("-----------------------------------------------------------------------------")
print("Number of test images:", len(test_dataset))


In [ ]:
for name, module in quantized_model.named_modules():
    if hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
        # Regular float layer
        print(f"{name} weight dtype: {module.weight.dtype}")
    elif hasattr(module, "weight") and callable(module.weight):
        # Quantized layer
        print(f"{name} weight dtype: {module.weight().dtype}")


In [ ]:
import torch
import torch.nn as nn
import time
from tqdm import tqdm

def evaluate_model_q(model, dataloader, device="cpu"):
    model.eval()
    model.to(device)  # quantized models usually on CPU

    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels.float().unsqueeze(1))
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).int() 

            correct += (predicted.view(-1) == labels.int().view(-1)).sum().item() 
            total += labels.numel()

    end_time = time.time()
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)

    print(f"\n✅ Accuracy: {accuracy:.2f}%")
    print(f"📉 Average Loss: {avg_loss:.4f}")
    print(f"⏱️ Evaluation Time: {end_time - start_time:.2f}s")

    return accuracy, avg_loss


In [ ]:
print ("Quantized model testing:")
evaluate_model_q(quantized_model, test_loader, device="cpu")

In [ ]:
# scripted_model = torch.jit.script(quantized_model)
# scripted_model.save("quantized_scripted_model.pt")
